# Personalized CS Education Agent — Ministral-3 Fine-tuning
## DATA 298B | MSDA Project II | Team 2 | Spring 2026

**Model family:** Mistral — Ministral-3 (December 2025 release)
- `mistralai/Ministral-3-3B-Instruct-2512` — 3–4B parameter slot
- `mistralai/Ministral-3-8B-Instruct-2512` — 7–8B parameter slot

**Evaluation metrics (CS-domain specific):**

| Metric | Purpose |
|--------|---------|
| ROUGE-L | Explanation coverage against reference answers |
| BLEU-4 | N-gram precision of generated CS explanations |
| BERTScore-F1 | Semantic correctness beyond word matching |
| Perplexity | Domain adaptation quality |
| CS-MMLU Subset | Domain knowledge: college CS, high school CS, computer security |
| HumanEval Pass@1 | Code generation correctness — gold standard for CS tutors |
| P50 Latency | Inference speed; team target < 200 ms |

**Hardware target:** Google Colab A100 (40 GB VRAM)

---
**Usage:** Set `MODEL_SIZE = "3B"` or `"8B"` in Cell 5, then run *Runtime → Run All*.

In [ ]:
import subprocess, torch, os

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU      : {gpu}")
    print(f"VRAM     : {vram:.1f} GB")
    if vram >= 38:
        print("A100 confirmed — full configuration enabled.")
    elif vram >= 14:
        print("Non-A100 GPU detected — reduce BATCH_SIZE to 2 if you encounter OOM errors.")
    else:
        print("Insufficient VRAM — A100 is strongly recommended for this notebook.")
else:
    raise RuntimeError(
        "No GPU found. Enable GPU runtime: Runtime > Change runtime type > GPU (A100)."
    )

In [ ]:
%%capture
# Library versions tested on Colab A100 (March 2026).
!pip install -q "transformers>=4.47.0" "peft>=0.13.2" "trl>=0.12.2"
!pip install -q "bitsandbytes>=0.44.1" "accelerate>=1.2.1" "datasets>=3.1.0"
!pip install -q "evaluate>=0.4.3" "rouge_score" "bert_score" "sacrebleu"
!pip install -q "matplotlib>=3.7" "seaborn" "pandas" "numpy" "scipy"
!pip install -q "sentencepiece" "protobuf" "nltk"
print("Package installation complete.")

In [ ]:
import os, json, time, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Optional

import torch
import transformers
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import (
    LoraConfig, get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel, TaskType,
)
from trl import SFTTrainer
from datasets import load_dataset, Dataset
import evaluate

import nltk
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

warnings.filterwarnings("ignore")

print(f"transformers : {transformers.__version__}")
print(f"torch        : {torch.__version__}")
print(f"CUDA device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  Set MODEL_SIZE to "3B" (faster, lighter) or "8B" (higher accuracy).
MODEL_SIZE = "3B"
# ─────────────────────────────────────────────────────────────────────────

MODEL_CONFIG = {
    "3B": {
        "model_id"   : "mistralai/Ministral-3-3B-Instruct-2512",
        "output_dir" : "./ministral_3b_cs_tutor_lora",
        "batch_size" : 8,
        "grad_accum" : 2,
        "max_steps"  : 500,
        "label"      : "Ministral-3B  (3-4B slot)",
    },
    "8B": {
        "model_id"   : "mistralai/Ministral-3-8B-Instruct-2512",
        "output_dir" : "./ministral_8b_cs_tutor_lora",
        "batch_size" : 4,
        "grad_accum" : 4,
        "max_steps"  : 500,
        "label"      : "Ministral-8B  (7-8B slot)",
    },
}

cfg        = MODEL_CONFIG[MODEL_SIZE]
MODEL_ID   = cfg["model_id"]
OUTPUT_DIR = cfg["output_dir"]
BATCH_SIZE = cfg["batch_size"]
GRAD_ACCUM = cfg["grad_accum"]
MAX_STEPS  = cfg["max_steps"]

# Hyper-parameters
LEARNING_RATE = 2e-4
MAX_SEQ_LEN   = 1024
LORA_RANK     = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
WARMUP_RATIO  = 0.03

# Dataset
DATASET_SIZE = 5000
TEST_SPLIT   = 0.10
SEED         = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# System prompt — scoped to CS tutoring only (Professor in a Box concept).
# Covers the domains evaluated in this project: algorithms, data structures,
# recursion, dynamic programming, OOP, debugging, and code analysis.
SYSTEM_PROMPT = (
    "You are a Computer Science professor and tutor. Your role is to explain "
    "CS concepts clearly and precisely to university-level students. Topics "
    "include algorithms (sorting, graph traversal, Dijkstra's, dynamic "
    "programming), data structures (trees, heaps, hash tables, linked lists), "
    "recursion, object-oriented programming, complexity analysis (Big-O), "
    "debugging strategies, and code correctness. Always provide step-by-step "
    "reasoning and working code examples in Python where appropriate."
)

print("=" * 62)
print(f"  Model  : {MODEL_ID}")
print(f"  Desc   : {cfg['label']}")
print(f"  Output : {OUTPUT_DIR}")
print(f"  Batch  : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective")
print(f"  Steps  : {MAX_STEPS}")
print("=" * 62)

In [ ]:
# Ministral-3 models are released under Apache 2.0 and are publicly accessible.
# A HuggingFace token is only needed if you hit rate-limit errors.
from huggingface_hub import login

HF_TOKEN = ""   # Paste your token here if needed: https://huggingface.co/settings/tokens

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace.")
else:
    print("No HF token provided — using public access (sufficient for Ministral-3).")

In [ ]:
# CS keyword list covers the topics defined in the project scope:
# algorithms, data structures, recursion, DP, OOP, debugging, complexity.
CS_KEYWORDS = [
    "algorithm", "data structure", "binary tree", "recursion", "sorting",
    "dynamic programming", "graph traversal", "python", "object-oriented",
    "machine learning", "neural network", "big-o", "hash table",
    "stack", "queue", "linked list", "inheritance", "polymorphism",
    "database", "sql", "operating system", "computer network",
    "debugging", "complexity", "pointer", "class definition",
    "function", "loop", "array", "string manipulation", "bit manipulation",
    "dijkstra", "breadth-first", "depth-first", "binary search",
    "merge sort", "quick sort", "heap", "graph", "tree traversal",
]

def is_cs_content(example):
    text = (example.get("text") or example.get("prompt") or "").lower()
    return any(kw in text for kw in CS_KEYWORDS) and len(text) > 300

def format_as_qa(text: str):
    sentences = [s.strip() for s in text.replace("\n", " ").split(". ") if s.strip()]
    if len(sentences) < 4:
        return None
    instruction = sentences[0] + ("?" if not sentences[0].endswith("?") else "")
    response    = ". ".join(sentences[1:]).strip()
    if len(instruction) < 20 or len(response) < 80:
        return None
    return {"instruction": instruction, "response": response}

print(f"Loading CS dataset — target {DATASET_SIZE:,} records from Cosmopedia ...")
records = []

try:
    raw = load_dataset(
        "HuggingFaceTB/cosmopedia", "web_samples_v2",
        split="train", streaming=True, trust_remote_code=True,
    )
    scanned = 0
    for ex in raw:
        scanned += 1
        if scanned % 20_000 == 0:
            print(f"  Scanned {scanned:,} -> {len(records):,} CS records collected ...")
        if is_cs_content(ex):
            fmt = format_as_qa(ex.get("text") or ex.get("prompt") or "")
            if fmt:
                records.append(fmt)
        if len(records) >= DATASET_SIZE:
            break
    print(f"Done: {len(records):,} CS records collected (scanned {scanned:,} total).")

except Exception as e:
    print(f"Cosmopedia stream failed ({e}). Falling back to iamtarun/python_code_instructions_18k_alpaca ...")
    fallback = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
    for ex in fallback:
        records.append({
            "instruction": ex.get("instruction", ""),
            "response"   : ex.get("output", ""),
        })
        if len(records) >= DATASET_SIZE:
            break
    print(f"Fallback loaded: {len(records):,} records.")

print(f"Dataset size: {len(records):,} CS Q&A pairs.")

In [ ]:
def build_prompt(instruction: str, response: str = "", training: bool = True) -> str:
    """Format using Mistral/Ministral instruction-tuning template with CS tutor persona."""
    body = (
        f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
        f"{instruction} [/INST] "
    )
    if training:
        return body + response + "</s>"
    return body

df = pd.DataFrame(records).dropna()
df = df[df["instruction"].str.len() > 20]
df = df[df["response"].str.len()    > 60]
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
df["text"] = df.apply(lambda r: build_prompt(r["instruction"], r["response"]), axis=1)

split_idx = int(len(df) * (1 - TEST_SPLIT))
train_df  = df[:split_idx].reset_index(drop=True)
test_df   = df[split_idx:].reset_index(drop=True)

train_ds = Dataset.from_pandas(train_df[["text", "instruction", "response"]])
test_ds  = Dataset.from_pandas(test_df [["text", "instruction", "response"]])

test_df.to_csv("test_dataset.csv", index=False)

print(f"Train split : {len(train_ds):,} samples")
print(f"Test split  : {len(test_ds):,} samples  (saved to test_dataset.csv)")
avg_toks = df["text"].str.len().mean() / 4
print(f"Avg tokens/sample (estimate): {avg_toks:.0f}")

In [ ]:
print(f"Loading tokenizer: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True, padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer loaded.")
print(f"  Vocab size : {tokenizer.vocab_size:,}")
print(f"  Pad token  : {tokenizer.pad_token!r}")
print(f"  EOS token  : {tokenizer.eos_token!r}")

In [ ]:
print(f"Loading model with 4-bit NF4 QLoRA: {MODEL_ID} ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",          # Normal Float 4 — optimal for QLoRA
    bnb_4bit_compute_dtype    = torch.bfloat16,  # native on A100
    bnb_4bit_use_double_quant = True,            # saves ~0.4 bits/parameter
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(model)

# Target the attention projection layers — consistent with team's LoRA config.
peft_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, peft_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters : {trainable:,}  ({100 * trainable / total:.2f}%)")
print(f"Total parameters     : {total:,}")
print(f"VRAM allocated       : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    optim                       = "paged_adamw_8bit",
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = 0.001,
    max_steps                   = MAX_STEPS,
    bf16                        = True,
    fp16                        = False,
    max_grad_norm               = 0.3,
    gradient_checkpointing      = True,
    logging_steps               = 25,
    evaluation_strategy         = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    report_to                   = "none",
    seed                        = SEED,
    dataloader_num_workers      = 2,
    remove_unused_columns       = False,
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    eval_dataset       = test_ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    peft_config        = peft_config,
    args               = training_args,
)

print(f"Starting fine-tuning: {MODEL_ID}")
print(f"  Train samples      : {len(train_ds):,}")
print(f"  Max steps          : {MAX_STEPS}")
print(f"  Effective batch    : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Estimated duration : 20-40 min on A100")
print("-" * 62)

t0           = time.time()
train_result = trainer.train()
elapsed      = time.time() - t0

print("-" * 62)
print(f"Training complete in {elapsed / 60:.1f} min")
print(f"  Final train loss : {train_result.training_loss:.4f}")
print(f"  Steps completed  : {train_result.global_step}")

In [ ]:
if trainer.state.log_history:
    log_df  = pd.DataFrame(trainer.state.log_history)
    fig, axs = plt.subplots(1, 2, figsize=(13, 4))

    train_l = log_df[log_df["loss"].notna()]      if "loss"      in log_df.columns else pd.DataFrame()
    eval_l  = log_df[log_df["eval_loss"].notna()] if "eval_loss" in log_df.columns else pd.DataFrame()

    if not train_l.empty:
        axs[0].plot(train_l["step"], train_l["loss"], "b-", lw=2)
        axs[0].set(xlabel="Step", ylabel="Loss",
                   title=f"Train Loss — Ministral-{MODEL_SIZE}")
        axs[0].grid(alpha=0.3)

    if not eval_l.empty:
        axs[1].plot(eval_l["step"], eval_l["eval_loss"], "r-", lw=2)
        axs[1].set(xlabel="Step", ylabel="Loss",
                   title=f"Eval Loss — Ministral-{MODEL_SIZE}")
        axs[1].grid(alpha=0.3)

    plt.suptitle(
        f"Ministral-{MODEL_SIZE} QLoRA Fine-tuning — CS Education Agent",
        fontsize=13
    )
    plt.tight_layout()
    plt.savefig(f"training_curves_{MODEL_SIZE}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved training_curves_{MODEL_SIZE}.png")

In [ ]:
final_dir = f"{OUTPUT_DIR}/final_checkpoint"
trainer.model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

print(f"LoRA adapters saved to: {final_dir}")
for fname in os.listdir(final_dir):
    sz = os.path.getsize(os.path.join(final_dir, fname))
    print(f"  {fname:<40} {sz / 1e6:>7.1f} MB")

del trainer
torch.cuda.empty_cache()
gc.collect()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
print("Loading fine-tuned model for evaluation ...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16,
)
finetuned_model = PeftModel.from_pretrained(
    base_model, final_dir, torch_dtype=torch.bfloat16,
)
finetuned_model.eval()
print("Fine-tuned model ready.")

print("Loading base model (no adapters) for comparison ...")
base_only = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16,
)
base_only.eval()
print("Base model ready.")
print(f"VRAM total: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
def generate(model, instruction: str, max_new_tokens: int = 256) -> str:
    """Run inference using the CS tutor prompt format."""
    prompt = build_prompt(instruction, training=False)
    enc    = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN - max_new_tokens,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens     = max_new_tokens,
            do_sample          = False,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.pad_token_id,
            eos_token_id       = tokenizer.eos_token_id,
        )
    new_ids = out[0][enc["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

# Quick sanity check with a representative CS question
Q = "Explain Dijkstra's algorithm and walk through an example with a small weighted graph"
print(f"Test question: {Q}\n")
print("Fine-tuned ->", generate(finetuned_model, Q)[:400], "...")
print()
print("Base only  ->", generate(base_only, Q)[:400], "...")

In [ ]:
# EVALUATION 1: ROUGE-L & BLEU-4
# Text-overlap metrics measuring how well the model covers reference explanations.
print("=" * 62)
print("EVAL 1/6 — ROUGE-L & BLEU-4  (Explanation Coverage)")
print("=" * 62)

rouge_m = evaluate.load("rouge")
bleu_m  = evaluate.load("bleu")

n_eval  = min(100, len(test_ds))
samples = test_ds.select(range(n_eval))

ft_preds, base_preds, refs = [], [], []

print(f"Generating responses for {n_eval} test samples ...")
for i, s in enumerate(samples):
    if i % 20 == 0:
        print(f"  {i}/{n_eval}")
    ft_preds.append(generate(finetuned_model, s["instruction"]))
    base_preds.append(generate(base_only,      s["instruction"]))
    refs.append(s["response"])

rouge_ft   = rouge_m.compute(predictions=ft_preds,   references=refs,                use_aggregator=True)
rouge_base = rouge_m.compute(predictions=base_preds, references=refs,                use_aggregator=True)
bleu_ft    = bleu_m.compute( predictions=ft_preds,   references=[[r] for r in refs])
bleu_base  = bleu_m.compute( predictions=base_preds, references=[[r] for r in refs])

eval_results = {
    "ROUGE-1" : {"Base": rouge_base["rouge1"], "FT": rouge_ft["rouge1"]},
    "ROUGE-L" : {"Base": rouge_base["rougeL"], "FT": rouge_ft["rougeL"]},
    "BLEU-4"  : {"Base": bleu_base["bleu"],    "FT": bleu_ft["bleu"]},
}

print(f"\n{'Metric':<12} {'Base':>10} {'Fine-tuned':>12} {'Delta':>10}")
print("-" * 46)
for m, v in eval_results.items():
    delta = ((v["FT"] - v["Base"]) / max(v["Base"], 1e-6)) * 100
    print(f"{m:<12} {v['Base']:>10.4f} {v['FT']:>12.4f} {delta:>+9.1f}%")

In [ ]:
# EVALUATION 2: BERTScore
# Measures semantic similarity between generated and reference explanations.
# A model can correctly explain recursion using different words than the reference —
# BLEU would penalise this; BERTScore captures conceptual correctness instead.
print("=" * 62)
print("EVAL 2/6 — BERTScore  (Semantic Accuracy)")
print("=" * 62)

bs_m = evaluate.load("bertscore")

print("Fine-tuned BERTScore ...")
bs_ft   = bs_m.compute(predictions=ft_preds,   references=refs,
                       model_type="distilbert-base-uncased", lang="en", batch_size=16)
print("Base model BERTScore ...")
bs_base = bs_m.compute(predictions=base_preds, references=refs,
                       model_type="distilbert-base-uncased", lang="en", batch_size=16)

for tag, src in [("BERTScore-P", "precision"), ("BERTScore-R", "recall"), ("BERTScore-F1", "f1")]:
    eval_results[tag] = {
        "Base": float(np.mean(bs_base[src])),
        "FT"  : float(np.mean(bs_ft[src])),
    }

print(f"\n{'Metric':<15} {'Base':>10} {'Fine-tuned':>12} {'Delta':>10}")
print("-" * 50)
for tag in ["BERTScore-P", "BERTScore-R", "BERTScore-F1"]:
    v = eval_results[tag]
    d = ((v["FT"] - v["Base"]) / max(v["Base"], 1e-6)) * 100
    print(f"{tag:<15} {v['Base']:>10.4f} {v['FT']:>12.4f} {d:>+9.1f}%")

In [ ]:
# EVALUATION 3: Perplexity
# Measures how well the model has adapted to CS educational language.
# Lower perplexity on the CS test set indicates stronger domain adaptation.
print("=" * 62)
print("EVAL 3/6 — Perplexity  (Domain Adaptation)")
print("=" * 62)

def calc_perplexity(mdl, texts: List[str], bs: int = 4) -> float:
    mdl.eval()
    total_loss, total_n = 0.0, 0
    for i in range(0, len(texts), bs):
        batch = texts[i:i + bs]
        enc   = tokenizer(batch, return_tensors="pt", truncation=True,
                          max_length=512, padding=True).to(mdl.device)
        with torch.no_grad():
            out = mdl(**enc, labels=enc["input_ids"])
        n = enc["attention_mask"].sum().item()
        total_loss += out.loss.item() * n
        total_n    += n
    return float(torch.exp(torch.tensor(total_loss / total_n)))

eval_texts = [s["text"] for s in samples][:50]

print("Computing fine-tuned perplexity ...")
ppl_ft   = calc_perplexity(finetuned_model, eval_texts)
print("Computing base-model perplexity ...")
ppl_base = calc_perplexity(base_only, eval_texts)

eval_results["Perplexity"] = {"Base": ppl_base, "FT": ppl_ft}

reduction = ((ppl_base - ppl_ft) / ppl_base) * 100
print(f"\n  Base model perplexity : {ppl_base:.2f}")
print(f"  Fine-tuned perplexity : {ppl_ft:.2f}")
print(f"  Reduction             : {reduction:+.1f}%  (lower is better)")

In [ ]:
# EVALUATION 4: MMLU Computer Science Subset
# Uses only the three CS-relevant MMLU tasks — not the generic full MMLU benchmark.
# This directly addresses the professor's feedback about using meaningful metrics.
print("=" * 62)
print("EVAL 4/6 — MMLU CS Subset  (Domain Knowledge)")
print("  Tasks: college_computer_science")
print("         high_school_computer_science")
print("         computer_security")
print("=" * 62)

CS_MMLU_TASKS = [
    "college_computer_science",
    "high_school_computer_science",
    "computer_security",
]

def mmlu_accuracy(mdl, task: str, n: int = 40) -> dict:
    try:
        ds = load_dataset("cais/mmlu", task, split="test", trust_remote_code=True)
        ds = ds.select(range(min(n, len(ds))))
    except Exception as e:
        print(f"  Could not load {task}: {e}")
        return {"accuracy": 0.0, "correct": 0, "total": 0}

    correct = 0
    for item in ds:
        q           = item["question"]
        choices     = item["choices"]
        ans_idx     = item["answer"]
        choices_str = "\n".join(f"{chr(65+i)}. {c}" for i, c in enumerate(choices))
        prompt      = f"{q}\n\n{choices_str}\n\nAnswer with the single letter A, B, C, or D:"
        resp        = generate(mdl, prompt, max_new_tokens=4).upper()
        pred        = next((ord(ch) - 65 for ch in resp[:5] if ch in "ABCD"), -1)
        if pred == ans_idx:
            correct += 1

    acc = correct / len(ds)
    print(f"  {task:<40} {acc:.1%}  ({correct}/{len(ds)})")
    return {"accuracy": acc, "correct": correct, "total": len(ds)}

print("\nFine-tuned model:")
mmlu_ft   = {t: mmlu_accuracy(finetuned_model, t, 40) for t in CS_MMLU_TASKS}

print("\nBase model:")
mmlu_base = {t: mmlu_accuracy(base_only, t, 40) for t in CS_MMLU_TASKS}

for prefix, src in [("FT", mmlu_ft), ("Base", mmlu_base)]:
    tot_c = sum(v["correct"] for v in src.values())
    tot_n = sum(v["total"]   for v in src.values())
    src["overall"] = {"accuracy": tot_c / tot_n if tot_n else 0.0, "correct": tot_c, "total": tot_n}

eval_results["MMLU-CS-Overall"] = {
    "Base": mmlu_base["overall"]["accuracy"],
    "FT"  : mmlu_ft  ["overall"]["accuracy"],
}
for t in CS_MMLU_TASKS:
    label = "MMLU-" + t.replace("_", " ").title()[:22]
    eval_results[label] = {
        "Base": mmlu_base[t]["accuracy"],
        "FT"  : mmlu_ft  [t]["accuracy"],
    }

print(f"\n  MMLU-CS overall  Base: {mmlu_base['overall']['accuracy']:.1%}  "
      f"Fine-tuned: {mmlu_ft['overall']['accuracy']:.1%}")

In [ ]:
# EVALUATION 5: HumanEval Pass@1
# Gold standard for code generation correctness.
# A CS tutor that cannot produce working code has limited value for students.
print("=" * 62)
print("EVAL 5/6 — HumanEval Pass@1  (Code Generation Correctness)")
print("=" * 62)

os.environ["HF_ALLOW_CODE_EVAL"] = "1"
code_eval_m = evaluate.load("code_eval")

N_PROBLEMS = 30  # 30 problems is sufficient for a meaningful Pass@1 estimate

try:
    he_ds  = load_dataset("openai_humaneval", split="test", trust_remote_code=True)
    he_sub = he_ds.select(range(N_PROBLEMS))
    print(f"Loaded {N_PROBLEMS} HumanEval problems.")

    def get_completions(mdl, dataset):
        preds = []
        for i, prob in enumerate(dataset):
            if i % 10 == 0:
                print(f"  Problem {i + 1}/{N_PROBLEMS}")
            inp = tokenizer(
                f"Complete the following Python function (return only the code):\n\n{prob['prompt']}",
                return_tensors="pt", truncation=True, max_length=512,
            ).to(mdl.device)
            with torch.no_grad():
                out = mdl.generate(
                    **inp, max_new_tokens=256, do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            gen = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
            preds.append([prob["prompt"] + gen])
        return preds

    refs_he = [[p["test"] + "\ncheck(" + p["entry_point"] + ")"] for p in he_sub]

    print("\nFine-tuned completions ...")
    preds_ft   = get_completions(finetuned_model, he_sub)
    print("Base model completions ...")
    preds_base = get_completions(base_only, he_sub)

    scores_ft,   _ = code_eval_m.compute(references=refs_he, predictions=preds_ft,   k=[1])
    scores_base, _ = code_eval_m.compute(references=refs_he, predictions=preds_base, k=[1])

    pass1_ft   = scores_ft.get("pass@1",   0.0)
    pass1_base = scores_base.get("pass@1", 0.0)

except Exception as e:
    print(f"HumanEval evaluation error: {e}")
    print("Falling back to 0.0 — re-run in an environment with code execution enabled.")
    pass1_ft   = 0.0
    pass1_base = 0.0

eval_results["HumanEval Pass@1"] = {"Base": pass1_base, "FT": pass1_ft}

delta = ((pass1_ft - pass1_base) / max(pass1_base, 1e-6)) * 100
print(f"\n  Base model  Pass@1 : {pass1_base:.1%}")
print(f"  Fine-tuned  Pass@1 : {pass1_ft:.1%}")
print(f"  Improvement        : {delta:+.1f}%")

In [ ]:
# EVALUATION 6: Inference Latency
# The team's deployment target is P50 latency < 200 ms on GPU.
# This cell measures P50, P95, and P99 latency across 50 representative CS questions.
print("=" * 62)
print("EVAL 6/6 — Inference Latency  (Team target: P50 < 200 ms)")
print("=" * 62)

LATENCY_QUESTIONS = [
    "What is Big-O notation?",
    "Explain the difference between a stack and a queue.",
    "How does merge sort work?",
    "What is dynamic programming?",
    "Describe Dijkstra's shortest path algorithm.",
    "What is recursion and when should you use it?",
    "Explain binary search.",
    "What is a hash table and how does it handle collisions?",
    "Describe depth-first search on a graph.",
    "What is the difference between a class and an object in OOP?",
] * 5  # 50 measurements

def measure_latency(mdl, questions):
    latencies = []
    for q in questions:
        t_start = time.perf_counter()
        generate(mdl, q, max_new_tokens=128)
        latencies.append((time.perf_counter() - t_start) * 1000)  # ms
    return np.array(latencies)

print("Measuring fine-tuned model latency (50 queries) ...")
lat_ft   = measure_latency(finetuned_model, LATENCY_QUESTIONS)
print("Measuring base model latency (50 queries) ...")
lat_base = measure_latency(base_only, LATENCY_QUESTIONS)

def lat_report(name, arr):
    print(f"  {name}:")
    print(f"    P50  : {np.percentile(arr, 50):.1f} ms")
    print(f"    P95  : {np.percentile(arr, 95):.1f} ms")
    print(f"    P99  : {np.percentile(arr, 99):.1f} ms")
    print(f"    Mean : {arr.mean():.1f} ms  |  Std: {arr.std():.1f} ms")

lat_report("Fine-tuned model", lat_ft)
lat_report("Base model",       lat_base)

target_ms = 200.0
p50_ft    = float(np.percentile(lat_ft, 50))
status    = "PASS" if p50_ft < target_ms else "FAIL"
print(f"\n  Team target (P50 < {target_ms:.0f} ms): {status}  (P50 = {p50_ft:.1f} ms)")

eval_results["P50 Latency (ms)"] = {
    "Base": float(np.percentile(lat_base, 50)),
    "FT"  : p50_ft,
}
eval_results["P95 Latency (ms)"] = {
    "Base": float(np.percentile(lat_base, 95)),
    "FT"  : float(np.percentile(lat_ft,  95)),
}

In [ ]:
# Full results table across all six evaluation metrics.
print("\n" + "=" * 70)
print(f"  COMPLETE RESULTS — Ministral-{MODEL_SIZE} | CS Education Agent")
print("=" * 70)

rows = []
for metric, v in eval_results.items():
    base, ft = v["Base"], v["FT"]
    if metric in ("Perplexity", "P50 Latency (ms)", "P95 Latency (ms)"):
        delta = ((base - ft) / max(base, 1e-6)) * 100
        note  = "lower is better"
    else:
        delta = ((ft - base) / max(base, 1e-6)) * 100
        note  = "higher is better"
    rows.append({
        "Metric"     : metric,
        "Base Model" : round(base, 4),
        "Fine-tuned" : round(ft, 4),
        "Improvement": f"{delta:+.1f}%",
        "Note"       : note,
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

os.makedirs("results", exist_ok=True)
summary_df.to_csv(f"results/eval_summary_{MODEL_SIZE}.csv", index=False)
with open(f"results/eval_results_{MODEL_SIZE}.json", "w") as fh:
    json.dump({
        "model_id"    : MODEL_ID,
        "model_size"  : MODEL_SIZE,
        "steps"       : MAX_STEPS,
        "dataset_size": DATASET_SIZE,
        "metrics"     : eval_results,
    }, fh, indent=2)

print(f"\nSaved: results/eval_summary_{MODEL_SIZE}.csv")
print(f"Saved: results/eval_results_{MODEL_SIZE}.json")

In [ ]:
# Four-panel figure suitable for Demo 2 slides and Workbook 2 report.
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    f"Ministral-{MODEL_SIZE} — CS Education Agent Evaluation\n"
    f"Base Model vs. QLoRA Fine-tuned ({DATASET_SIZE:,} CS training samples)",
    fontsize=14, fontweight="bold", y=1.01,
)

# Panel 1: Core metrics bar chart
ax1  = axes[0, 0]
CORE = ["ROUGE-L", "BLEU-4", "BERTScore-F1", "MMLU-CS-Overall", "HumanEval Pass@1"]
CORE = [m for m in CORE if m in eval_results]
x    = np.arange(len(CORE))
w    = 0.35
b_sc = [eval_results[m]["Base"] for m in CORE]
f_sc = [eval_results[m]["FT"]   for m in CORE]

b1 = ax1.bar(x - w/2, b_sc, w, label="Base Model",         color="#5B9BD5", alpha=0.9, edgecolor="white")
b2 = ax1.bar(x + w/2, f_sc, w, label="Fine-tuned (QLoRA)", color="#ED7D31", alpha=0.9, edgecolor="white")
ax1.set_xticks(x)
ax1.set_xticklabels(CORE, rotation=15, ha="right", fontsize=9)
ax1.set(ylabel="Score", title="Core Metrics — Base vs Fine-tuned", ylim=(0, 1.15))
ax1.legend(fontsize=9)
for bar in list(b1) + list(b2):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7.5)

# Panel 2: Improvement percentages
ax2  = axes[0, 1]
imps = []
for m in CORE:
    base, ft = eval_results[m]["Base"], eval_results[m]["FT"]
    imps.append(((ft - base) / max(base, 1e-6)) * 100)
colors2 = ["#27AE60" if v >= 0 else "#E74C3C" for v in imps]
hb = ax2.barh(CORE, imps, color=colors2, alpha=0.85, edgecolor="white")
ax2.axvline(0, color="black", lw=0.8)
ax2.set(xlabel="Improvement (%)", title="Improvement from Fine-tuning")
for bar, val in zip(hb, imps):
    xpos = val + (0.5 if val >= 0 else -0.5)
    ha   = "left" if val >= 0 else "right"
    ax2.text(xpos, bar.get_y() + bar.get_height() / 2,
             f"{val:+.1f}%", ha=ha, va="center", fontsize=9, fontweight="bold")

# Panel 3: Perplexity
ax3 = axes[1, 0]
if "Perplexity" in eval_results:
    pv  = eval_results["Perplexity"]
    pb  = ax3.bar(["Base Model", "Fine-tuned"], [pv["Base"], pv["FT"]],
                  color=["#5B9BD5", "#ED7D31"], alpha=0.9, width=0.4, edgecolor="white")
    for bar, val in zip(pb, [pv["Base"], pv["FT"]]):
        ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"{val:.1f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
    red = ((pv["Base"] - pv["FT"]) / pv["Base"]) * 100
    ax3.text(0.5, 0.93, f"{red:.1f}% reduction — improved domain adaptation",
             transform=ax3.transAxes, ha="center", fontsize=10, color="#27AE60", fontweight="bold")
    ax3.set(ylabel="Perplexity",
            title="Perplexity on CS Test Set (lower = better domain adaptation)")

# Panel 4: MMLU subcategory breakdown
ax4    = axes[1, 1]
mmlu_keys = [k for k in eval_results if k.startswith("MMLU-") and k != "MMLU-CS-Overall"]
if mmlu_keys:
    m_x    = np.arange(len(mmlu_keys))
    m_base = [eval_results[k]["Base"] for k in mmlu_keys]
    m_ft   = [eval_results[k]["FT"]   for k in mmlu_keys]
    ax4.bar(m_x - 0.2, m_base, 0.4, label="Base",    color="#5B9BD5", alpha=0.9, edgecolor="white")
    ax4.bar(m_x + 0.2, m_ft,   0.4, label="Fine-tuned", color="#ED7D31", alpha=0.9, edgecolor="white")
    labels = [k.replace("MMLU-", "").replace(" ", "\n") for k in mmlu_keys]
    ax4.set_xticks(m_x)
    ax4.set_xticklabels(labels, fontsize=9)
    ax4.set(ylabel="Accuracy", title="MMLU CS Subcategory Accuracy", ylim=(0, 1.0))
    ax4.legend(fontsize=9)
    ax4.axhline(0.25, color="gray", lw=0.8, linestyle="--", label="Random baseline")

plt.tight_layout()
plt.savefig(f"eval_charts_{MODEL_SIZE}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved eval_charts_{MODEL_SIZE}.png")

In [ ]:
import zipfile
from google.colab import files

zip_name = f"ministral_{MODEL_SIZE}_cs_education_agent.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk("results"):
        for fname in fnames:
            fp = os.path.join(root, fname)
            zf.write(fp)
            print(f"  + {fp}")
    for fname in [f"training_curves_{MODEL_SIZE}.png", f"eval_charts_{MODEL_SIZE}.png"]:
        if os.path.exists(fname):
            zf.write(fname)
            print(f"  + {fname}")
    for fname in os.listdir(final_dir):
        fp = os.path.join(final_dir, fname)
        if os.path.isfile(fp):
            zf.write(fp, os.path.join("lora_adapters", fname))
            print(f"  + lora_adapters/{fname}")

print(f"\nPackage ready: {zip_name}")
files.download(zip_name)

## Fine-tuning Complete — Next Steps

### What you now have
- Fine-tuned Ministral-{MODEL_SIZE} adapted to CS tutoring (algorithms, DSA, recursion, OOP, debugging)
- LoRA adapter weights saved locally and included in the download zip
- Six evaluation metrics covering text quality, semantic accuracy, domain knowledge, code correctness, and latency
- Charts ready for Demo 2 slides

### Immediate actions
1. Re-run with `MODEL_SIZE = "8B"` to produce the second model variant.
2. Share your numbers (ROUGE-L, BLEU-4, BERTScore-F1, MMLU-CS, Pass@1, P50 latency) with teammates.
3. Add results to Workbook 2 — Sections 4.3 (model architecture), 4.4 (training), 4.5 (evaluation), 6.1 (results), 6.3 (comparison table).
4. Upload LoRA adapters to the team Google Drive folder for the presentation.

### Why these metrics satisfy the professor's feedback
| Metric | Justification for CS Education Agent |
|--------|--------------------------------------|
| HumanEval Pass@1 | Directly measures whether generated code is functionally correct |
| CS-MMLU Subset | Domain-specific knowledge test — not generic MMLU |
| BERTScore-F1 | Captures conceptual correctness even when wording differs from reference |
| ROUGE-L | Measures coverage of key teaching points in explanations |
| Perplexity | Quantifies how well the model adapted to CS educational language |
| P50 Latency | Validates real-time tutoring feasibility against the team's 200 ms target |